# Poisson Equation: Plain Python Finite Elements and Firedrake

This notebook contains two ways of solving the Poisson model problem.

Part A is an implementation of a Python finite element method for the Poisson equation on the unit square `by hand', i.e. not using Firedrake:
1. we build a triangular mesh ourselves,
2. we use $P1$ nodal basis functions on each triangle,
3. we assemble the global stiffness matrix and load vector element by element,
4. we impose homogeneous Dirichlet boundary conditions by modifying the algebraic system,
5. and we solve the sparse linear system with `scipy.sparse.linalg.spsolve`.

This follows the general steps of the lecture notes found at `https://finite-element.github.io/`.

Part B instead does the Firedrake implementation, and collects the main optimal-control steps:

1. a forward Poisson solve in Firedrake,  
2. a manufactured-solution convergence study for the forward solver,  
3. a manual adjoint Poisson solve,  
4. gradient computation for the reduced functional,  
5. a Taylor test for gradient correctness.

The model problem we want to solve is

$$
-\Delta u = m \quad \text{in } \Omega, \qquad u=0 \quad \text{on } \partial\Omega,
$$

and the weak form is

$$
\int_\Omega \nabla u \cdot \nabla v \, dx
=
\int_\Omega m v \, dx
\qquad \forall v \in H_0^1(\Omega).
$$

So the notebook now shows both:
- a lower-level finite element solve done without Firedrake, and
- a higher-level Firedrake implementation that connects directly to the dissertation theory.


## Part A. Poisson numerically without Firedrake

Here we write a finite element method in Python. The idea as follows:

We triangulate the unit square $\Omega=(0,1)^2$, choose the standard piecewise linear $P1$ nodal basis, and solve the discrete variational problem

$$
\int_\Omega \nabla u_h \cdot \nabla v_h \, dx
=
\int_\Omega m\, v_h \, dx
\qquad \forall v_h \in V_h^0.
$$

In matrix form this becomes

$$
K \mathbf{u} = \mathbf{f},
$$

where the entries are assembled from the local element contributions

$$
K_{ij}^{(K)} = \int_K \nabla \phi_i \cdot \nabla \phi_j\,dx,
\qquad
f_i^{(K)} = \int_K \phi_i\,m\,dx.
$$

This follows what is done in the course doing the key steps
1. choose the basis,
2. compute local element integrals,
3. assemble into a global sparse matrix,
4. identify boundary nodes,
5. impose Dirichlet conditions,
6. solve the resulting linear system.


### Manufactured solution for the plain Python finite element solver

To check that the implementation is correct, we use a manufactured exact solution.

We choose

$$
u_{\mathrm{exact}}(x,y)=\sin(\pi x)\sin(\pi y),
$$

so that

$$
-\Delta u_{\mathrm{exact}} = 2\pi^2 \sin(\pi x)\sin(\pi y),
$$

and so that the source term is

$$
m(x,y)=2\pi^2 \sin(\pi x)\sin(\pi y).
$$

We can now solve the finite element problem on a sequence of successively refined meshes and compute the $L^2$-error

$$
\|u_h-u_{\mathrm{exact}}\|_{L^2(\Omega)}.
$$

Since we are using $P1$ finite elements and the exact solution is smooth, we expect the $L^2$-error to decay like $O(h^2)$.

A good conceptual point to keep in mind is this:

- in the plain Python section, we are making the finite element machinery explicit;
- in the Firedrake section later, the same variational structure is expressed much more compactly through UFL forms and automated assembly.


In [2]:
import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as spla
import math

def make_unit_square_tri_mesh(nx, ny=None):
    """
    Build a very simple structured triangular mesh of the unit square.

    Parameters
    ----------
    nx, ny : int
        Number of rectangular subdivisions in the x- and y-directions.
        Each rectangle is then split into two triangles.

    Returns
    -------
    vertices : ndarray, shape (N, 2)
        Coordinates of all mesh vertices.
    triangles : ndarray, shape (T, 3)
        Vertex indices for each triangle.
    """
    if ny is None:
        ny = nx

    xs = np.linspace(0.0, 1.0, nx + 1)
    ys = np.linspace(0.0, 1.0, ny + 1)

    # Vertex ordering: loop over rows in y, and within each row over x.
    vertices = np.array([(x, y) for y in ys for x in xs], dtype=float)

    def vertex_id(i, j):
        return j * (nx + 1) + i

    triangles = []
    for j in range(ny):
        for i in range(nx):
            v00 = vertex_id(i, j)
            v10 = vertex_id(i + 1, j)
            v01 = vertex_id(i, j + 1)
            v11 = vertex_id(i + 1, j + 1)

            # Split each square into two triangles.
            # The ordering is chosen to keep a consistent orientation.
            triangles.append([v00, v10, v11])
            triangles.append([v00, v11, v01])

    return vertices, np.array(triangles, dtype=int)


# Reference triangle basis functions:
#   phi_1 = 1 - xi - eta
#   phi_2 = xi
#   phi_3 = eta
#
# Their gradients with respect to the reference coordinates (xi, eta)
# are constant on the reference triangle.
REFERENCE_GRADS = np.array([
    [-1.0, -1.0],
    [ 1.0,  0.0],
    [ 0.0,  1.0],
])


# A simple degree-2 quadrature rule on the reference triangle.
# The reference triangle has vertices (0,0), (1,0), (0,1) and area 1/2.
# This three-point rule is exact for polynomials up to degree 2.
QUAD_POINTS = np.array([
    [1.0 / 6.0, 1.0 / 6.0],
    [2.0 / 3.0, 1.0 / 6.0],
    [1.0 / 6.0, 2.0 / 3.0],
])

QUAD_WEIGHTS = np.array([
    1.0 / 6.0,
    1.0 / 6.0,
    1.0 / 6.0,
])


def p1_basis_values(xi, eta):
    """
    Evaluate the three P1 basis functions on the reference triangle.
    """
    return np.array([
        1.0 - xi - eta,
        xi,
        eta,
    ])


def boundary_nodes(vertices, tol=1.0e-12):
    """
    Return the indices of vertices lying on the boundary of the unit square.

    This mirrors the course idea that homogeneous Dirichlet conditions can
    be enforced once we know which basis functions are attached to boundary
    nodes.
    """
    x = vertices[:, 0]
    y = vertices[:, 1]

    on_boundary = (
        (x < tol) | (x > 1.0 - tol) |
        (y < tol) | (y > 1.0 - tol)
    )
    return np.flatnonzero(on_boundary)


def assemble_poisson_p1(nx, rhs_function):
    """
    Assemble the global finite element system A u = b for

        -Δu = rhs_function   in Ω,
             u = 0           on ∂Ω,

    using P1 elements on a structured triangular mesh.

    Parameters
    ----------
    nx : int
        Number of square subdivisions in each direction.
        (The mesh has 2*nx*nx triangles.)
    rhs_function : callable
        Function of two variables, rhs_function(x, y).

    Returns
    -------
    vertices : ndarray
        Mesh vertex coordinates.
    triangles : ndarray
        Triangle connectivity.
    A : scipy.sparse.csr_matrix
        Global stiffness matrix before/after boundary modification.
    b : ndarray
        Global load vector.
    """
    vertices, triangles = make_unit_square_tri_mesh(nx)

    n_vertices = len(vertices)

    # LIL format is convenient for assembly by repeated insertion.
    A = sp.lil_matrix((n_vertices, n_vertices))
    b = np.zeros(n_vertices)

    for tri in triangles:
        coords = vertices[tri]

        # Affine map from reference triangle to physical triangle:
        #   x = x0 + J X
        x0 = coords[0]
        J = np.column_stack((coords[1] - coords[0], coords[2] - coords[0]))
        detJ = np.linalg.det(J)
        abs_detJ = abs(detJ)
        area = 0.5 * abs_detJ

        # For affine P1 elements, the physical gradients are constant on
        # each cell and obtained by the chain rule.
        invJ = np.linalg.inv(J)
        grad_phys = REFERENCE_GRADS @ invJ

        # Local stiffness matrix:
        #   K_e(i,j) = int_K grad(phi_i)·grad(phi_j) dx
        #
        # Since the gradients are constant on a P1 triangle, this integral
        # reduces to area * dot(grad_i, grad_j).
        local_A = area * (grad_phys @ grad_phys.T)

        # Local load vector:
        #   f_e(i) = int_K phi_i * rhs dx
        #
        # We evaluate this using quadrature on the reference triangle.
        local_b = np.zeros(3)

        for (xi, eta), w in zip(QUAD_POINTS, QUAD_WEIGHTS):
            phi = p1_basis_values(xi, eta)

            # Map quadrature point to the physical triangle.
            xq = x0 + J @ np.array([xi, eta])

            # Pull back the integral:
            #   int_K (...) dx = int_{Khat} (...) |det J| dX
            local_b += w * abs_detJ * rhs_function(xq[0], xq[1]) * phi

        # Local-to-global assembly.
        for a, Arow in enumerate(tri):
            b[Arow] += local_b[a]
            for bcol, Acol in enumerate(tri):
                A[Arow, Acol] += local_A[a, bcol]

    # Impose homogeneous Dirichlet boundary conditions.
    #
    # This follows the standard implementation idea in the course:
    # assemble first, then overwrite the rows corresponding to boundary nodes.
    bdry = boundary_nodes(vertices)
    b[bdry] = 0.0

    for i in bdry:
        A.rows[i] = [i]
        A.data[i] = [1.0]

    return vertices, triangles, A.tocsr(), b


def solve_poisson_p1(nx):
    """
    Solve the manufactured Poisson problem with P1 finite elements.

    We use the exact solution
        u_exact = sin(pi x) sin(pi y),
    so that
        rhs = 2 pi^2 sin(pi x) sin(pi y).

    Returns
    -------
    result : dict
        Dictionary containing the mesh, coefficients, and error data.
    """
    u_exact = lambda x, y: math.sin(math.pi * x) * math.sin(math.pi * y)
    rhs = lambda x, y: 2.0 * math.pi**2 * math.sin(math.pi * x) * math.sin(math.pi * y)

    vertices, triangles, A, b = assemble_poisson_p1(nx, rhs)
    uh = spla.spsolve(A, b)

    # Compute an L2 error by quadrature over each triangle.
    error_sq = 0.0

    for tri in triangles:
        coords = vertices[tri]
        x0 = coords[0]
        J = np.column_stack((coords[1] - coords[0], coords[2] - coords[0]))
        abs_detJ = abs(np.linalg.det(J))

        uh_local = uh[tri]

        for (xi, eta), w in zip(QUAD_POINTS, QUAD_WEIGHTS):
            phi = p1_basis_values(xi, eta)
            xq = x0 + J @ np.array([xi, eta])

            uh_q = np.dot(uh_local, phi)
            ue_q = u_exact(xq[0], xq[1])

            error_sq += w * abs_detJ * (uh_q - ue_q)**2

    l2_error = math.sqrt(error_sq)

    return {
        "vertices": vertices,
        "triangles": triangles,
        "uh": uh,
        "l2_error": l2_error,
        "h": 1.0 / nx,
        "min_u": float(np.min(uh)),
        "max_u": float(np.max(uh)),
    }


def manufactured_poisson_convergence_p1(mesh_sizes=(4, 8, 16, 32)):
    """
    Run a convergence study for the plain Python P1 finite element solver.
    """
    rows = []
    previous_error = None

    for nx in mesh_sizes:
        result = solve_poisson_p1(nx)
        error = result["l2_error"]

        if previous_error is None:
            rate = None
        else:
            rate = math.log(previous_error / error, 2.0)

        rows.append((nx, result["h"], error, rate))
        previous_error = error

    return rows


In [3]:
# Run the plain Python P1 finite element convergence study and print a summary.
p1_rows = manufactured_poisson_convergence_p1(mesh_sizes=(4, 8, 16, 32))

print("Plain Python P1 finite element convergence test")
print()
print(f"{'nx':>6} {'h':>14} {'L2 error':>18} {'rate':>10}")
print("-" * 52)

for nx, h, err, rate in p1_rows:
    rate_str = "-" if rate is None else f"{rate:.4f}"
    print(f"{nx:6d} {h:14.6f} {err:18.10e} {rate_str:>10}")

sample = solve_poisson_p1(16)
print()
print("One sample solve (nx = 16):")
print(f"number of vertices = {len(sample['vertices'])}")
print(f"number of triangles = {len(sample['triangles'])}")
print(f"min(u_h) = {sample['min_u']:.6f}")
print(f"max(u_h) = {sample['max_u']:.6f}")


Plain Python P1 finite element convergence test

    nx              h           L2 error       rate
----------------------------------------------------
     4       0.250000   7.5963325311e-02          -
     8       0.125000   2.0409490653e-02     1.8961
    16       0.062500   5.2008558841e-03     1.9724
    32       0.031250   1.3065718315e-03     1.9930

One sample solve (nx = 16):
number of vertices = 289
number of triangles = 512
min(u_h) = 0.000000
max(u_h) = 0.996797


The observed convergence rates are close to $2$ in the $L^2$-norm, which is exactly what we expect for a smooth solution with $P1$ finite elements. So, now we have solved the forward Poisson equation without Firedrake.


Now, we use Firedrake to solve the same problem. We work entirely with standard Firedrake finite-element objects:
- `Mesh` and `FunctionSpace` for the discrete setting,
- `TrialFunction`/`TestFunction` for weak forms,
- Function for the computed state, adjoint, control, and gradient,
- assemble/solve for evaluation and linear solves.

In [4]:
from firedrake import *
import math
import numpy as np

We have that the forward weak problem is

$$
b(u,v) = (m,v)_{L^2(\Omega)} \qquad \forall v \in V,
$$

with $V = H_0^1(\Omega)$ and
$$
b(u,v) = \int_\Omega \nabla u \cdot \nabla v \, dx.
$$

In the code below:

- `V = FunctionSpace(mesh, "CG", degree)` is the finite-dimensional approximation of the state space,
- `u = TrialFunction(V)` and `v = TestFunction(V)` represent the unknown and test function,
- `a = inner(grad(u), grad(v)) * dx` is the bilinear form,
- `L = m * v * dx` is the right-hand side,
- `DirichletBC(...)` imposes the homogeneous boundary condition.


## B.1 Forward Poisson solve

In [5]:
def solve_forward_poisson(mesh, m_expr=None, m_function=None, degree=1):
    '''
    Solve the forward problem

        -Delta u = m   in Omega,
               u = 0   on boundary.

    Inputs
    ------
    mesh : Firedrake mesh
        Computational mesh.
    m_expr : UFL expression, optional
        Analytic expression for the control/source term.
    m_function : Firedrake Function, optional
        Discrete control already stored as a Function.
    degree : int
        Polynomial degree for the continuous Galerkin space.

    Returns
    -------
    V : FunctionSpace
        Finite-element space used for both state and control here.
    m : Function
        The discrete control/source term.
    u_sol : Function
        The computed state.
    '''
    # In this notebook we use a continuous Galerkin space CG(degree).
    # This is the discrete counterpart of the state space V = H_0^1(Omega).
    V = FunctionSpace(mesh, "CG", degree)

    # Trial and test functions for the weak formulation
    #     int grad(u)·grad(v) dx = int m v dx.
    u = TrialFunction(V)
    v = TestFunction(V)

    # Allow either:
    # 1. a ready-made discrete control m_function, or
    # 2. a symbolic/UFL expression m_expr to be interpolated.
    if m_function is not None:
        m = m_function
    else:
        m = Function(V, name="control")
        m.interpolate(m_expr)

    # Bilinear form and right-hand side of the weak problem.
    a = inner(grad(u), grad(v)) * dx
    L = m * v * dx

    # Homogeneous Dirichlet boundary condition u = 0 on ∂Omega.
    bc = DirichletBC(V, 0.0, "on_boundary")

    # Solve the linear system and store the result in u_sol.
    u_sol = Function(V, name="state")
    solve(a == L, u_sol, bcs=bc)

    return V, m, u_sol


In [6]:
# A quick one-off forward solve with a smooth Gaussian-type source term.
# This is mainly a sanity check before doing more systematic verification.
mesh = UnitSquareMesh(32, 32)
x, y = SpatialCoordinate(mesh)

m_expr = exp(-50*((x - 0.5)**2 + (y - 0.5)**2))
V, m, u = solve_forward_poisson(mesh, m_expr=m_expr, degree=1)

# Report the range of the computed state.
# This is not a full error check, but it confirms that the solve produced
# a finite, nontrivial solution.
u_min = float(u.dat.data_ro.min())
u_max = float(u.dat.data_ro.max())

print(f"u range: min = {u_min:.6e}, max = {u_max:.6e}")


u range: min = 0.000000e+00, max = 1.616821e-02


Before we check gradients or adjoints, we should verify that the forward problem is behaving sensibly. The quick test below uses a smooth Gaussian-type control $m$ and solves the Poisson problem once. Printing the range of the state is not a full verification, but it is a convenient sanity check that the solve completed and produced a nontrivial solution.


## B.2 Manufactured-solution convergence study

In [7]:
def manufactured_poisson_convergence(mesh_sizes=(8, 16, 32, 64), degree=1):
    '''
    Manufactured-solution convergence study.

    Choose
        u_exact = sin(pi x) sin(pi y),
    so that
        -Delta u_exact = 2*pi^2*sin(pi x)sin(pi y).

    Because u_exact satisfies the homogeneous Dirichlet boundary condition,
    it is a convenient exact solution for testing the forward solver.
    '''
    errors = []
    hs = []

    for n in mesh_sizes:
        # Build a uniform n x n mesh and the corresponding CG space.
        mesh = UnitSquareMesh(n, n)
        x, y = SpatialCoordinate(mesh)
        V = FunctionSpace(mesh, "CG", degree)

        # Exact solution and matching source term.
        u_exact_expr = sin(math.pi * x) * sin(math.pi * y)
        m_expr = 2 * math.pi**2 * sin(math.pi * x) * sin(math.pi * y)

        # Weak form of the forward problem:
        #     int grad(u_h)·grad(v) dx = int m v dx.
        u_trial = TrialFunction(V)
        v = TestFunction(V)

        a = inner(grad(u_trial), grad(v)) * dx
        L = m_expr * v * dx

        bc = DirichletBC(V, 0.0, "on_boundary")

        # Solve for the discrete state u_h.
        u_h = Function(V, name="u_h")
        solve(a == L, u_h, bcs=bc)

        # Interpolate the exact solution into the same FE space so that
        # errornorm compares like with like on the mesh.
        u_exact = Function(V, name="u_exact")
        u_exact.interpolate(u_exact_expr)

        # L2 error and mesh size.
        err = errornorm(u_exact, u_h, norm_type="L2")
        h = 1.0 / n

        hs.append(h)
        errors.append(err)

    print("   n          h              L2 error         observed rate")
    print("-------------------------------------------------------------")
    for i, (h, err) in enumerate(zip(hs, errors)):
        if i == 0:
            rate_str = "   -"
        else:
            # Experimental convergence rate:
            # log(err_{k-1}/err_k) / log(h_{k-1}/h_k).
            rate = math.log(errors[i-1] / err) / math.log(hs[i-1] / h)
            rate_str = f"{rate:7.4f}"
        n = int(round(1.0 / h))
        print(f"{n:4d}   {h:10.4e}   {err:14.6e}   {rate_str}")

    return hs, errors


In [8]:
# Run the convergence study.
# For CG1 elements and this smooth exact solution, the L2 errors
# should decrease at approximately second order.
hs, errors = manufactured_poisson_convergence()


   n          h              L2 error         observed rate
-------------------------------------------------------------
   8   1.2500e-01     6.277592e-03      -
  16   6.2500e-02     1.617041e-03    1.9569
  32   3.1250e-02     4.073878e-04    1.9889
  64   1.5625e-02     1.020450e-04    1.9972


A manufactured solution gives an exact state $u_{\text{exact}}$ and a matching source term $m$.
This lets us measure the error
$$
\|u_h-u_{\text{exact}}\|_{L^2(\Omega)}
$$
as the mesh is refined.

For continuous piecewise linear elements (`CG1`) on a smooth solution, we expect approximately **second-order convergence in the $L^2$ norm**.
That makes this a good check that the forward discretisation has been set up correctly.


## B.3 Objective, adjoint solve, and gradient

Once the forward state $u(m)$ is computed, the adjoint $\lambda$ is obtained from

$$
\int_\Omega \nabla \lambda \cdot \nabla v \, dx
=
\int_\Omega (u-d) v \, dx
\qquad \forall v \in V.
$$

This is the weak form of
$$
-\Delta \lambda = u-d.
$$

After that, the reduced gradient in the control space $L^2(\Omega)$ is simply

$$
\nabla \tilde J(m)=\lambda + \alpha m.
$$

So the implementation pattern is:

1. solve the forward problem for $u$,
2. solve the adjoint problem for $\lambda$,
3. combine them to form the gradient.


In [9]:
def objective(u, m, d, alpha):
    """
    Full objective functional

        J(u,m) = 1/2 ||u-d||^2 + (alpha/2) ||m||^2.
    """
    tracking_term = 0.5 * assemble((u - d)**2 * dx)
    regularisation_term = 0.5 * float(alpha) * assemble(m**2 * dx)
    return tracking_term + regularisation_term


def solve_adjoint_poisson(mesh, u, d, degree=1):
    '''
    Solve the adjoint problem

        -Delta lambda = u - d   in Omega,
                 lambda = 0      on boundary.

    In weak form:
        int grad(lambda)·grad(v) dx = int (u-d) v dx
        for all test functions v.
    '''
    V = FunctionSpace(mesh, "CG", degree)

    # Trial and test functions for the adjoint solve.
    lam_trial = TrialFunction(V)
    v = TestFunction(V)

    # The adjoint bilinear form is the same Poisson operator as in the
    # forward problem. Only the right-hand side changes.
    a_adj = inner(grad(lam_trial), grad(v)) * dx
    L_adj = (u - d) * v * dx

    bc = DirichletBC(V, 0.0, "on_boundary")

    lam = Function(V, name="adjoint")
    solve(a_adj == L_adj, lam, bcs=bc)

    return lam


def compute_gradient(mesh, m_function, d_function, alpha, degree=1):
    '''
    Return the state u, the adjoint lambda, and the reduced gradient

        g = lambda + alpha*m,

    interpreted here as the L2-gradient in the control space.
    '''
    # Step 1: solve the forward problem to obtain u(m).
    V, m, u = solve_forward_poisson(mesh, m_function=m_function, degree=degree)

    # Step 2: solve the adjoint problem driven by the state misfit u-d.
    lam = solve_adjoint_poisson(mesh, u, d_function, degree=degree)

    # Step 3: form the gradient using the analytic formula derived
    # from the reduced functional.
    g = Function(V, name="gradient")
    g.interpolate(lam + float(alpha) * m)

    return u, lam, g


In [10]:
# Build a test configuration for the optimal-control quantities.
mesh = UnitSquareMesh(40, 40)
V = FunctionSpace(mesh, "CG", 1)
x, y = SpatialCoordinate(mesh)

# Example control m.
m = Function(V, name="control")
m.interpolate(exp(-30*((x - 0.3)**2 + (y - 0.4)**2)))

# Desired state d.
d = Function(V, name="desired_state")
d.interpolate(sin(math.pi * x) * sin(math.pi * y))

# Tikhonov regularisation parameter.
alpha = 1e-3

# Compute the state, adjoint, and gradient for this control.
u, lam, g = compute_gradient(mesh, m, d, alpha, degree=1)

Jval = objective(u, m, d, alpha)
print(f"Objective value J = {Jval:.8e}")
print(f"Gradient L2 norm = {norm(g):.8e}")


Objective value J = 1.21341725e-01
Gradient L2 norm = 2.47796263e-02


## B.4 Taylor test for gradient correctness

In [11]:
import math

def reduced_objective(mesh, m_function, d_function, alpha, degree=1):
    """
    Evaluate the reduced functional \tilde J(m) = J(u(m), m).

    Computationally, this means:
    1. solve the forward problem for u(m),
    2. plug the resulting state into the full objective.
    """
    V, m, u = solve_forward_poisson(mesh, m_function=m_function, degree=degree)
    return objective(u, m, d_function, alpha)


def directional_derivative(mesh, m_function, direction, d_function, alpha, degree=1):
    """
    Compute the directional derivative d\tilde J(m; direction)
    by taking the L2 inner product of the gradient with the direction.
    """
    u, lam, g = compute_gradient(mesh, m_function, d_function, alpha, degree=degree)
    return assemble(g * direction * dx)


def taylor_test(mesh, m_function, direction, d_function, alpha, degree=1, eps_list=None):
    """
    Perform a first-order Taylor test.

    If the gradient is correct, then the remainder
        |J(m+eps*dm) - J(m) - eps*dJ(m;dm)|
    should behave like O(eps^2).
    """
    if eps_list is None:
        eps_list = [1e-1, 5e-2, 2.5e-2, 1.25e-2, 6.25e-3]

    # Baseline reduced functional value and directional derivative.
    J0 = reduced_objective(mesh, m_function, d_function, alpha, degree=degree)
    dJ = directional_derivative(mesh, m_function, direction, d_function, alpha, degree=degree)

    label_width = 18
    eps_width = 12
    val_width = 22
    rate_width = 12

    print(f"{'J(m)':<{label_width}} = {J0: .10e}")
    print(f"{'dJ(m; direction)':<{label_width}} = {dJ: .10e}")
    print()

    header = (
        f"{'epsilon':>{eps_width}} | "
        f"{'|J(m+eps*dm)-J(m)|':>{val_width}} | "
        f"{'first-order remainder':>{val_width}} | "
        f"{'~rate':>{rate_width}}"
    )
    print(header)
    print("-" * len(header))

    prev_rem = None
    for eps in eps_list:
        # Build the perturbed control m + eps * direction.
        m_eps = Function(m_function.function_space(), name="m_eps")
        m_eps.interpolate(m_function + eps * direction)

        # Evaluate the reduced functional at the perturbed control.
        J_eps = reduced_objective(mesh, m_eps, d_function, alpha, degree=degree)

        # Zeroth-order difference and first-order Taylor remainder.
        diff0 = abs(J_eps - J0)
        rem1 = abs(J_eps - J0 - eps * dJ)

        rate_str = ""
        if prev_rem is not None and rem1 > 0:
            # Because eps is halved each step, the observed order can be
            # estimated using log(prev/current)/log(2).
            rate = math.log(prev_rem / rem1) / math.log(2.0)
            rate_str = f"{rate:5.10f}"

        print(
            f"{eps:>{eps_width}.3e} | "
            f"{diff0:>{val_width}.10e} | "
            f"{rem1:>{val_width}.10e} | "
            f"{rate_str:>{rate_width}}"
        )

        prev_rem = rem1


In [12]:
# Choose a smooth perturbation direction δm.
# In a Taylor test, this is the direction along which we perturb the control.
direction = Function(V, name="direction")
direction.interpolate(cos(2*math.pi*x) * sin(math.pi*y))

# Run the Taylor test.
# The first-order remainder should converge like O(eps^2) if the
# gradient implementation is correct.
taylor_test(mesh, m, direction, d, alpha, degree=1)


J(m)               =  1.2134172534e-01
dJ(m; direction)   =  5.2535360717e-03

     epsilon |     |J(m+eps*dm)-J(m)| |  first-order remainder |        ~rate
-----------------------------------------------------------------------------
   1.000e-01 |       5.2724588135e-04 |       1.8922741837e-06 |             
   5.000e-02 |       2.6314987213e-04 |       4.7306854594e-07 | 2.0000000000
   2.500e-02 |       1.3145666893e-04 |       1.1826713672e-07 | 1.9999999971
   1.250e-02 |       6.5698767680e-05 |       2.9566784492e-08 | 1.9999999848
   6.250e-03 |       3.2841992144e-05 |       7.3916961712e-09 | 1.9999999906


The Taylor test is the standard practical check that the derivative and gradient have been implemented correctly.

If the directional derivative
$$
d\tilde J(m;\delta m)
$$
is correct, then the first-order remainder
$$
\left|\tilde J(m+\varepsilon \delta m)-\tilde J(m)-\varepsilon d\tilde J(m;\delta m)\right|
$$
should decay like $O(\varepsilon^2)$ as $\varepsilon \to 0$.

So in the output table below, the most important quantity is the **first-order remainder** and its observed rate.
A rate close to $2$ is the main sign that the gradient implementation is correct.


## 5. Notes

- If the adjoint gradient is implemented correctly, the **first-order remainder**
  in the Taylor test should decrease approximately like $O(\varepsilon^2)$.
- The forward and adjoint solves here are both written **manually from the weak forms**,
  which makes the connection to the theory transparent.
- This notebook is a natural numerical continuation of the dissertation sections on:
  - the forward weak form,
  - the reduced functional,
  - the adjoint equation,
  - and the $L^2$-gradient formula.
- A natural next extension would be to reorganise the code into reusable functions for:
  - state solve,
  - adjoint solve,
  - reduced objective,
  - reduced gradient,
  - and Taylor-test verification.


## Suggested interpretation

At this point, the notebook demonstrates the full basic Poisson optimal-control workflow:

- solve the forward PDE,
- verify the forward solve with a convergence study,
- solve the adjoint PDE,
- assemble the reduced gradient,
- verify the gradient with a Taylor test.

That is exactly the numerical bridge from the theory chapter to the implementation chapter of the dissertation.
